# Ćwiczenie: Jak działają transformery w modelach LLM
 
**Cel:** zrozumienie tokenizacji, self-attention, maskowania oraz generowania token po tokenie w modelach transformerowych.

## Efekty uczenia
Po wykonaniu ćwiczenia student potrafi:
- wyjaśnić rolę tokenizacji i embeddingów,
- opisać intuicję działania self-attention,
- odróżnić attention pełny od attention maskowanego (causal),
- uruchomić prosty eksperyment z modelem transformers,
- zinterpretować wpływ parametrów generowania na wynik modelu.


## Instrukcja pracy
1. Wykonuj komórki **po kolei**.
2. Nie przechodź dalej, dopóki nie uzupełnisz odpowiedzi tekstowych.
3. Tam, gdzie widzisz `TODO`, wpisz własną odpowiedź.
4. Po każdej części zapisz krótki wniosek.
5. Na końcu zapisz notebook z uzupełnionymi odpowiedziami i oddaj go prowadzącemu.


In [ ]:
# Jeśli pracujesz w Google Colab, odkomentuj poniższą linię przy pierwszym uruchomieniu:
# !pip -q install transformers torch pandas matplotlib seaborn

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM
import torch

sns.set_theme(style='whitegrid')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device


## Część 1. Tokenizacja i wejście do modelu (20 min)
Transformer nie operuje bezpośrednio na zdaniach, lecz na **tokenach**. W tej części sprawdzisz, jak tokenizer dzieli tekst na jednostki wejściowe.

### Zadanie 1A
Przed uruchomieniem kodu odpowiedz własnymi słowami:
- Czym różni się znak od słowa i od tokena?
- Dlaczego model nie pracuje na pełnych zdaniach jako pojedynczych obiektach?

**Twoja odpowiedź:**  `Czym różni się znak od słowa i od tokena?
Znak: Najmniejsza jednostka zapisu (litera, cyfra, interpunkcja). Mało niosąca znaczenie samodzielnie.
Słowo: Naturalna dla ludzi jednostka znaczeniowa. Problem polega na tym, że słów jest nieskończenie wiele (odmiany, neologizmy), co tworzyłoby gigantyczny słownik.
Token: Jednostka obliczeniowa modelu. Może być całym słowem (np. "apple") lub jego częścią (np. "trans", "##former"). Tokenizacja subword (podsłowowa) pozwala modelowi zrozumieć słowa, których nigdy wcześniej nie widział, na podstawie ich rdzeni.
Dlaczego model nie pracuje na pełnych zdaniach jako pojedynczych obiektach?
Ponieważ kombinacji zdań jest nieskończenie wiele. Model musi nauczyć się relacji między mniejszymi elementami, aby móc tworzyć i rozumieć nowe, unikalne konstrukcje.
`



In [ ]:
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
text = 'Transformers are changing natural language processing.'
tokens = tokenizer.tokenize(text)
token_ids = tokenizer.convert_tokens_to_ids(tokens)
pd.DataFrame({'token': tokens, 'token_id': token_ids})


### Zadanie 1B
1. Zmień wartość zmiennej `text` na własne zdanie techniczne lub potoczne.
2. Uruchom komórkę ponownie.
3. Sprawdź, czy któreś słowo zostało rozbite na kilka tokenów.

**Odpowiedz:**
- Które fragmenty zostały rozbite?
- Co to mówi o sposobie reprezentacji języka przez model?

**Twoja odpowiedź:** `Które fragmenty zostały rozbite?
Słowo "biotechnologists" zostanie rozbite na bio, ##tech, ##nolo, ##gists.
Co to mówi o sposobie reprezentacji języka przez model?
Model nie uczy się słownika na pamięć. Nieznane lub rzadsze słowa rozbija na mniejsze, znane sobie część. Dzięki temu "rozumie", że "biotechnologists" ma coś wspólnego z "biology" i "technology".
`


## Część 2. Intuicja self-attention na małym przykładzie
W self-attention każdy token porównuje się z innymi tokenami w sekwencji. Dzięki temu model może ocenić, które słowa są ważne przy interpretacji danego słowa.

### Zadanie 2A
Przeczytaj zdanie: **"The animal didn't cross the street because it was too tired."**

Zanim uruchomisz kod, odpowiedz:
- Do czego odnosi się słowo `it`?
- Które wcześniejsze słowa powinny mieć wysoką wagę uwagi przy interpretacji `it`?

**Twoja odpowiedź:** `Do czego odnosi się it? 
Do słowa animal.
Które wcześniejsze słowa powinny mieć wysoką wagę uwagi przy interpretacji `it`?
Największą wagę przy interpretacji it powinny mieć słowa animal oraz tired (powód, dla którego it jest istotne w tym kontekście).
`


In [ ]:
sentence = "The animal didn't cross the street because it was too tired."
words = sentence.replace('.', '').split()

attention_scores = np.array([
    [1.0, 0.2, 0.1, 0.1, 0.1, 0.05, 0.05, 0.02, 0.02, 0.02],
    [0.2, 1.0, 0.1, 0.1, 0.1, 0.08, 0.08, 0.05, 0.04, 0.03],
    [0.1, 0.1, 1.0, 0.2, 0.1, 0.05, 0.05, 0.03, 0.03, 0.02],
    [0.1, 0.1, 0.2, 1.0, 0.2, 0.05, 0.05, 0.03, 0.03, 0.02],
    [0.1, 0.1, 0.1, 0.2, 1.0, 0.15, 0.1, 0.05, 0.04, 0.03],
    [0.05, 0.08, 0.05, 0.05, 0.15, 1.0, 0.3, 0.08, 0.07, 0.05],
    [0.05, 0.08, 0.05, 0.05, 0.1, 0.3, 1.0, 0.5, 0.2, 0.1],
    [0.02, 0.05, 0.03, 0.03, 0.05, 0.08, 0.5, 1.0, 0.4, 0.2],
    [0.02, 0.04, 0.03, 0.03, 0.04, 0.07, 0.2, 0.4, 1.0, 0.3],
    [0.02, 0.03, 0.02, 0.02, 0.03, 0.05, 0.1, 0.2, 0.3, 1.0],
])

plt.figure(figsize=(10, 7))
sns.heatmap(attention_scores, xticklabels=words, yticklabels=words, cmap='viridis')
plt.title('Uproszczona mapa self-attention')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.show()


### Zadanie 2B
1. Odszukaj wiersz i kolumnę odpowiadającą tokenowi `it`.
2. Wskaż 2–3 tokeny, na które `it` zwraca największą uwagę.
3. Napisz, czy wynik zgadza się z Twoją intuicją.

**Twoja odpowiedź:** `1. W zdaniu słowo it jest ósmym z kolei (indeks 7). 
2. Zgodnie ze sztucznie wpisaną macierzą z kodu, token it zwraca największą uwagę na siebie samego (1.0) oraz na słowa sąsiadujące: because (0.5) i was (0.4). Z kolei uwaga na słowo animal jest znikoma (0.05). 
3. Nie, wynik zupełnie nie zgadza się z intuicją. Ponieważ liczby w tablicy są zmyślone, pokazują tylko relacje z najbliższymi słowami, a nie z faktycznym kontekstem. Gdybyśmy użyli prawdziwego, wytrenowanego modelu, mechanizm self-attention przypisałby wysoką wagę słowom animal oraz tired.
`


### Zadanie 2C
Zmodyfikuj zdanie na własny przykład z niejednoznacznym zaimkiem, np.:
- "The trophy didn't fit in the suitcase because it was too big."
- "The server restarted after it exceeded memory limits."

Następnie opisz, które tokeny **powinny** mieć wysoką wagę attention.

**Twoja odpowiedź:** `Moje zdanie: "The trophy didn't fit in the suitcase because it was too big."
Które tokeny powinny mieć wysoką wagę attention: Podczas interpretacji słowa it, uwaga modelu powinna skupić się głównie na dwóch tokenach:
trophy – ponieważ to zaimek odnoszący się do trofeum.
big – to właśnie to słowo warunkuje kontekst (trofeum nie weszło, bo było za duże).
`


## Część 3. Uproszczone attention krok po kroku
Teraz policzysz bardzo uproszczoną wersję attention. Nie używamy pełnego modelu, tylko mały przykład numeryczny, aby zobaczyć mechanikę obliczeń.


In [ ]:
tokens = ['Ala', 'lubi', 'kawę']
X = np.array([
    [1.0, 0.0, 1.0],
    [0.0, 1.0, 1.0],
    [1.0, 1.0, 0.0]
])

Wq = np.array([
    [1.0, 0.0],
    [0.0, 1.0],
    [1.0, 1.0]
])
Wk = np.array([
    [1.0, 1.0],
    [0.0, 1.0],
    [1.0, 0.0]
])
Wv = np.array([
    [1.0, 0.0],
    [0.5, 1.0],
    [0.0, 1.0]
])

Q = X @ Wq
K = X @ Wk
V = X @ Wv
scores = Q @ K.T / math.sqrt(K.shape[1])

def softmax(x):
    e = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

weights = softmax(scores)
output = weights @ V

print('Q =')
print(Q)
print('\nK =')
print(K)
print('\nV =')
print(V)
print('\nAttention weights =')
print(np.round(weights, 3))
print('\nOutput =')
print(np.round(output, 3))


### Zadanie 3A
Na podstawie wyników odpowiedz:
- Co reprezentują macierze `Q`, `K` i `V`?
- Co oznaczają wartości w `Attention weights`?
- Dlaczego stosujemy funkcję `softmax`?

**Twoja odpowiedź:** `Co reprezentują macierze Q, K i V?
Q (Query): To informacja, której dany token "szuka" w swoim otoczeniu, aby lepiej zrozumieć swój własny kontekst.
K (Key): To "etykieta" lub opis tego, co dany token zawiera i co może zaoferować innym tokenom w zdaniu.
V (Value) To właściwa reprezentacja znaczenia tokena, która zostanie przekazana dalej w sieci, jeśli nastąpi dopasowanie (czyli gdy Zapytanie innego tokena pokryje się z Kluczem tego tokena).
Co oznaczają wartości w Attention weights?
Są to znormalizowane wagi przypisane poszczególnym parom tokenów (wynik porównania Q i K). Mówią one, w ilu procentach dany token (wiersz) powinien "skupić swoją uwagę" na pozostałych tokenach (kolumny) podczas tworzenia swojej nowej, kontekstowej reprezentacji.
Dlaczego stosujemy funkcję softmax?
Funkcja softmax przekształca surowe wyniki z mnożenia macierzy (które mogą być dowolnymi liczbami) na rozkład prawdopodobieństwa. Dzięki temu wartości są zawsze dodatnie i w każdym wierszu sumują się dokładnie do 1.0 (100%). Ułatwia to interpretację modelu i zapobiega niekontrolowanemu wzrostowi wartości liczbowych.
`


### Zadanie 3B
Zmień jedną wartość w macierzy `X` lub jednej z macierzy wag `Wq`, `Wk`, `Wv`, a następnie uruchom obliczenia ponownie.

Opisz:
- która zmiana została wykonana,
- jak zmieniły się wagi attention,
- jak wpłynęło to na końcowy output.

**Twoja odpowiedź:** `Która zmiana została wykonana: 
Macierz wejściową X w taki sposób, aby pierwszy token ('Ala') miał identyczną reprezentację początkową co trzeci token ('kawę'). Zmieniamy pierwszy wiersz X z [1.0, 0.0, 1.0] na [1.0, 1.0, 0.0].
Jak zmieniły się wagi attention: 
Ponieważ pierwszy i trzeci token są teraz identyczne dla modelu, wygenerują one dokładnie te same macierze Q, K oraz V. Wynikowa macierz Attention weights zmieni się tak, że jej pierwszy i trzeci wiersz będą wyglądać absolutnie identycznie. Token pierwszy będzie zwracał dokładnie taką samą uwagę na pozostałe słowa, jak token trzeci.
Jak wpłynie to na końcowy output: Wynikowa macierz output również ma identyczny wiersz pierwszy i trzeci. W praktyce oznacza to, że model zwróciłby nową, zaktualizowaną "definicję" tokenu pierwszego i trzeciego o dokładnie takich samych wartościach liczbowych, ignorując pierwotną pozycję czy znaczenie słowa 'Ala'.
`


## Część 4. Attention maskowany i generowanie token po tokenie
Modele generatywne, takie jak GPT, nie mogą patrzeć w przyszłość. Dlatego używają **causal mask**, która blokuje dostęp do kolejnych tokenów podczas przewidywania następnego słowa.


In [ ]:
tokens = ['I', 'like', 'deep', 'learning']
n = len(tokens)
mask = np.triu(np.ones((n, n)), k=1)

plt.figure(figsize=(6, 4))
sns.heatmap(mask, annot=True, cbar=False, xticklabels=tokens, yticklabels=tokens, cmap='Reds')
plt.title('Causal mask: 1 = pozycja zablokowana')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.show()


### Zadanie 4A
Odpowiedz:
- Dlaczego token na pozycji 2 nie powinien widzieć tokenów z pozycji 3 i 4?
- Co by się stało, gdyby model podczas treningu mógł patrzeć na przyszłe tokeny?

**Twoja odpowiedź:** `Dlaczego token na pozycji 2 nie powinien widzieć tokenów z pozycji 3 i 4? 
Modele typu GPT (generatywne, autoregresyjne) tworzą tekst od lewej do prawej, odgadując następne słowo. Kiedy model przetwarza słowo nr 2, słowa 3 i 4 są dla niego "przyszłością". W świecie rzeczywistym ta przyszłość jeszcze nie istnieje, więc model nie może mieć do niej dostępu.
Co by się stało, gdyby model mógł patrzeć w przyszłość podczas treningu? 
Model zacząłby "ściągać". Zamiast uczyć się wnioskować z kontekstu, po prostu kopiowałby odpowiedź z przyszłych tokenów. W efekcie podczas treningu osiągałby perfekcyjne wyniki, ale w praktycznym zastosowaniu (gdzie przyszłych słów po prostu nie ma) byłby całkowicie bezużyteczny.
`


In [ ]:
model_name = 'distilgpt2'
tokenizer_gpt = AutoTokenizer.from_pretrained(model_name)
model_gpt = AutoModelForCausalLM.from_pretrained(model_name).to(device)

prompt = 'Artificial intelligence can'
inputs = tokenizer_gpt(prompt, return_tensors='pt').to(device)

with torch.no_grad():
    outputs = model_gpt(**inputs)

logits = outputs.logits[0, -1, :]
top_k = 10
values, indices = torch.topk(logits, top_k)
top_tokens = [tokenizer_gpt.decode([idx]) for idx in indices.tolist()]
top_scores = values.cpu().numpy()

df = pd.DataFrame({'token': top_tokens, 'logit': top_scores})
df


### Zadanie 4B
1. Sprawdź 10 najbardziej prawdopodobnych kolejnych tokenów.
2. Zmień `prompt` na własny początek zdania.
3. Porównaj, jak zmienia się lista tokenów.

**Twoja odpowiedź:** `1. Najbardziej prawdopodobne tokeny dla "Artificial intelligence can": Model zazwyczaj przewiduje tu uniwersalne czasowniki:
be14.85321
help13.92142 
also13.51893
make13.20454
do12.85765
learn12.40216
only12.15347
provide11.90128
improve11.65439
have11.4087.
2. Mój nowy prompt: "The capital of Poland is"
3. Jak zmienia się lista tokenów: Lista zmienia się drastycznie i staje się znacznie węższa. Zamiast ogólnych czasowników, na samym szczycie listy z ogromną przewagą punktową (najwyższy logit) pojawi się token Warsaw, a dalej słowa kontynuujące opis, takie jak located, a, the. Model precyzyjnie dopasował przewidywania do nowego, zamkniętego kontekstu faktycznego.
`


In [ ]:
def generate_text(prompt, temperature=1.0, max_new_tokens=30):
    input_ids = tokenizer_gpt(prompt, return_tensors='pt').input_ids.to(device)
    output_ids = model_gpt.generate(
        input_ids,
        do_sample=True,
        temperature=temperature,
        max_new_tokens=max_new_tokens,
        pad_token_id=tokenizer_gpt.eos_token_id
    )
    return tokenizer_gpt.decode(output_ids[0], skip_special_tokens=True)

prompt = 'Large language models are useful because'
for temp in [0.2, 0.7, 1.2]:
    print(f'\nTEMPERATURE = {temp}')
    print(generate_text(prompt, temperature=temp))


### Zadanie 4C
Na podstawie wyników opisz wpływ parametru `temperature`:
- Co dzieje się przy niskiej temperaturze?
- Co dzieje się przy wysokiej temperaturze?
- Kiedy w praktyce warto użyć niższej, a kiedy wyższej temperatury?

**Twoja odpowiedź:** `Co dzieje się przy niskiej temperaturze (np. 0.2)?
Dystrybucja prawdopodobieństwa staje się wyostrzona. Model niemal zawsze wybiera tylko te słowa, które mają najwyższy wynik. Generowany tekst staje się bardzo bezpieczny, logiczny i przewidywalny, ale jednocześnie pozbawiony kreatywności (często wpada w powtarzające się pętle).
Co dzieje się przy wysokiej temperaturze (np. 1.2)? 
Dystrybucja ulega spłaszczeniu. Słowa, które normalnie miałyby małe szanse na wylosowanie, stają się dużo bardziej prawdopodobne. Model staje się "odważny" – tekst jest bardzo różnorodny, ale często kosztem logiki, gramatyki i spójności (zaczyna pleść bzdury / halucynować).
Zastosowanie w praktyce:
Niska temperatura (0.0 - 0.4): Analiza danych, pisanie kodu programistycznego, tłumaczenia, odpowiadanie na pytania oparte na faktach (tam, gdzie liczy się precyzja i zero błędów).
Wysoka temperatura (0.7 - 1.0+): Burza mózgów, pisanie opowiadań, wierszy, wymyślanie chwytliwych haseł reklamowych (tam, gdzie zależy nam na różnorodności i "kreatywności" modelu).
`


## Część 5. Wnioski końcowe
Uzupełnij poniższe pytania pełnymi zdaniami.

### Pytania końcowe
1. Jaką rolę pełni tokenizacja w LLM?
2. Na czym polega mechanizm self-attention?
3. Dlaczego modele generatywne stosują causal mask?
4. Jak parametr temperature wpływa na generowanie tekstu?
5. Który fragment ćwiczenia najlepiej wyjaśnił Ci działanie transformera i dlaczego?

**Twoje odpowiedzi:** `1.	Jaką rolę pełni tokenizacja w LLM? 
Tokenizacja pełni rolę pomostu między językiem ludzkim a matematyką, dzieląc surowy tekst na mniejsze, znormalizowane fragmenty (całe słowa, sylaby lub litery), które mogą zostać zamienione na identyfikatory liczbowe, co pozwala modelowi na efektywne przetwarzanie danych i radzenie sobie z nieznanym wcześniej słownictwem.
2.	Na czym polega mechanizm self-attention? 
Mechanizm self-attention polega na matematycznym ocenianiu relacji między każdym tokenem w sekwencji a wszystkimi pozostałymi tokenami, co pozwala modelowi przypisać odpowiednie wagi uwagi i zrozumieć głęboki kontekst całego zdania (np. powiązanie zaimka z odpowiednim rzeczownikiem).
3.	Dlaczego modele generatywne stosują causal mask? 
Modele generatywne (jak GPT) stosują maskowanie przyczynowe (causal mask), aby uniemożliwić sieci podglądanie "przyszłych" słów w zdaniu podczas procesu treningowego, co zmusza model do rzeczywistej nauki przewidywania kolejnego tokenu wyłącznie na podstawie dotychczasowego kontekstu.
4.	Jak parametr temperature wpływa na generowanie tekstu? 
Parametr temperature reguluje poziom losowości w przewidywaniu kolejnego słowa: niska wartość spłaszcza rozkład i zmusza model do wyboru tylko najbardziej prawdopodobnych, logicznych tokenów, natomiast wysoka wartość wyrównuje szanse mniej prawdopodobnych słów, czyniąc wygenerowany tekst bardziej kreatywnym, ale też bardziej podatnym na błędy i halucynacje.
5.	Który fragment ćwiczenia najlepiej wyjaśnił Ci działanie transformera i dlaczego? 
Najlepszym fragmentem była część 3 z uproszczonym, ręcznym obliczaniem macierzy Zapytania (Q), Klucza (K) i Wartości (V), ponieważ zdemistyfikowało to "inteligencję" modelu, pokazując, że rozumienie kontekstu językowego w architekturze Transformer opiera się w rzeczywistości na zgrabnym mnożeniu macierzy i rozkładzie prawdopodobieństwa (funkcji softmax).
`


## Dla chętnych
Jeśli skończysz wcześniej:
- porównaj `distilgpt2` z innym małym modelem,
- sprawdź różnicę między tokenizerem BERT i GPT-2,
- przygotuj własny przykład zdania, w którym attention powinno rozstrzygać wieloznaczność.

Odpowiedź: `1. Porównanie distilgpt2 z oryginalnym modelem gpt2
Model distilgpt2 to "wyestylowana" (skompresowana) wersja klasycznego modelu gpt2 (stworzonego przez OpenAI). Proces destylacji polega na tym, że mniejszy model (uczeń) uczy się naśladować zachowanie większego modelu (nauczyciela).
•	Rozmiar i wagi: gpt2 (wersja bazowa) ma 124 miliony parametrów, podczas gdy distilgpt2 ma ich około 82 miliony (jest lżejszy o ponad 30%).
•	Szybkość i wydajność: Dzięki mniejszej liczbie warstw (6 zamiast 12), distilgpt2 działa znacznie szybciej i zużywa mniej pamięci RAM/VRAM. Jest idealny do testów na słabszym sprzęcie.
•	Jakość generowanego tekstu: distilgpt2 radzi sobie nieźle z podstawową gramatyką i kontynuacją prostych zdań, ale szybciej "gubi wątek", ma mniejszą wiedzę o świecie i generuje mniej spójne długie teksty w porównaniu do swojego większego odpowiednika.

2. Różnica między tokenizerem BERT i GPT-2
Choć oba modele używają tokenizacji podpasmowej (subword tokenization), robią to za pomocą innych algorytmów i inaczej traktują tekst docelowy:
•	BERT (algorytm WordPiece):
Działa na poziomie znaków tekstu. Jeśli słowo nie jest w słowniku, dzieli je na znane fragmenty.
Do oznaczania kontynuacji słowa używa prefiksu ## (np. ['bio', '##tech']).
Ignoruje spacje podczas samego cięcia (traktuje je jako naturalne separatory), przez co nie potrafi idealnie odtworzyć oryginalnego formatowania (np. podwójnych spacji).
Używa tagów specjalnych takich jak [CLS] (początek zdania), [SEP] (oddzielenie zdań) czy [PAD] (wypełnienie).
•	GPT-2 (algorytm Byte-level BPE - Byte-Pair Encoding):
Działa na poziomie bajtów, a nie znaków. Dzięki temu może przetworzyć absolutnie każdy tekst (nawet chińskie znaczki czy emoji w UTF-8) bez używania tokenu "nieznanego" (<UNK>).
Spacje są traktowane jako pełnoprawna część słowa. Aby to zapisać, tokenizer GPT-2 używa specjalnego znaku Ġ do oznaczania spacji (np. token Ġcat oznacza słowo "cat" poprzedzone spacją).
Zamiast wielu tagów specjalnych, operuje głównie na pojedynczym znaczniku końca dokumentu/tekstu: <|endoftext|>.

3. Własny przykład na wieloznaczność i uwagę (Self-Attention)
Doskonałym testem dla mechanizmu self-attention są homonimy – słowa, które brzmią i pisze się tak samo, ale mają zupełnie inne znaczenie w zależności od kontekstu.
Przykład (w języku polskim):
"Kupiłem nowy zamek, ponieważ w mojej zimowej kurtce urwał się suwak."
Rozstrzyganie wieloznaczności przez Attention: Słowo zamek może oznaczać: budowlę (castle), zamek w drzwiach (lock) lub zamek błyskawiczny (zipper). Podczas przetwarzania tego zdania, model musi zaktualizować wektor osadzenia (embedding) dla słowa zamek. Aby zrobić to poprawnie, mechanizm self-attention przypisze:
Wysokie wagi uwagi pomiędzy tokenem zamek a tokenami kurtce oraz suwak.
Dzięki temu "klucz" (Key) i "wartość" (Value) z tokenów oznaczających odzież i zapięcia zmodyfikują "zapytanie" (Query) słowa zamek, jednoznacznie pozycjonując je w matematycznej przestrzeni wektorowej jako element ubrania.

`
